# Урок 23. Запросы, фильтры и сортировка

9 класс · III четверть

[⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/index.ipynb) · [← Урок 22](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-22.ipynb) · [Урок 24 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-24.ipynb)

---

Выборка данных по условию. Составные условия. Сортировка результата. Конструктор запросов Access: условия, И/ИЛИ, итоги, вычисляемые поля; режим SQL.

In [ ]:
#@title 🚀 Регистрация { display-mode: "form" }
#@markdown Впиши свои данные и запусти ячейку (Shift+Enter).
#@markdown Если заводил секрет `INFORMATIKA` и включил к нему доступ
#@markdown (🔑 на панели слева) — поля можно оставить пустыми.
#@markdown
#@markdown Colab спросит разрешение на доступ к аккаунту — нажми
#@markdown «Разрешить». Так проверка понимает, чья это работа.
ФИО = "" #@param {type:"string"}
#@markdown Класс — с буквой, например 9А
Класс = "" #@param {type:"string"}

import importlib, urllib.request
for адрес in ("https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/lib/schoolinf.py",
              "https://cdn.jsdelivr.net/gh/LeksssGray/informatika-7-11@main/lib/schoolinf.py"):
    try:
        urllib.request.urlretrieve(адрес, "schoolinf.py")
        break
    except Exception as ошибка:
        print("⚠️ Не скачалась библиотека проверки:", ошибка)
else:
    raise SystemExit("Проверь интернет и запусти эту ячейку ещё раз. "
                     "Не помогло — покажи это сообщение учителю.")
import schoolinf as si
importlib.reload(si)
si.start(lesson="09-23", name=ФИО, klass=Класс)

## Разбираемся

### Разговор с базой

Таблицы мы разложили, ключи расставили. Осталось главное: доставать
из базы то, что нужно. Для этого есть язык запросов — **SQL**.

Он устроен непривычно. В Python вы пишете, **как** получить ответ:
заведи счётчик, пройди по списку, сравни, добавь. В SQL пишут,
**что** нужно получить, а как — забота базы.

```sql
SELECT фамилия FROM читатели WHERE класс = '9А';
```

Всё. Перебор, сравнение, сборка результата — не ваша работа.

### Из чего состоит запрос

| Часть | Что задаёт | Обязательна |
|---|---|---|
| `SELECT` | какие поля показать | да |
| `FROM` | из какой таблицы | да |
| `WHERE` | условие отбора | нет |
| `ORDER BY` | порядок сортировки | нет |
| `LIMIT` | сколько строк вернуть | нет |

Порядок частей менять нельзя: сначала `SELECT`, потом `FROM`,
потом `WHERE`, и только потом `ORDER BY`.

```sql
SELECT название, цена       -- что
FROM товары                 -- откуда
WHERE цена > 100            -- какие строки
ORDER BY цена DESC          -- в каком порядке
LIMIT 3;                    -- сколько
```

`*` вместо списка полей означает «все поля»: `SELECT * FROM товары`.
В учебных примерах так пишут постоянно, в настоящих программах —
почти никогда: лишние поля тянутся по сети зря.

### Условия

В `WHERE` работают знакомые сравнения и новые слова:

| Условие | Что отбирает |
|---|---|
| `цена > 100` | дороже ста |
| `категория = 'хлеб'` | точное совпадение (кавычки одинарные!) |
| `цена BETWEEN 50 AND 100` | в промежутке, границы включаются |
| `категория IN ('хлеб', 'молоко')` | из списка |
| `название LIKE 'Мо%'` | начинается с «Мо» |
| `скидка IS NULL` | поле не заполнено |

Условия соединяются словами `AND`, `OR`, `NOT` — ровно как
в Python, только по-английски.

> Два подвоха. Равенство в SQL записывается **одним** знаком `=`,
> а не двумя. И текст берётся в **одинарные** кавычки: `'9А'`.

В `LIKE` символ `%` заменяет любое число символов, а `_` — ровно
один. Это те же маски файлов, что были в седьмом классе,
только значки другие.

### Сортировка

```sql
ORDER BY цена          -- по возрастанию (это по умолчанию)
ORDER BY цена DESC     -- по убыванию
ORDER BY категория, цена DESC   -- сначала по категории, внутри — по цене
```

`ASC` — по возрастанию, `DESC` — по убыванию. Первое можно
не писать.

### Подсчёты

| Функция | Что считает |
|---|---|
| `COUNT(*)` | количество строк |
| `SUM(поле)` | сумму |
| `AVG(поле)` | среднее |
| `MIN`, `MAX` | минимум и максимум |

```sql
SELECT COUNT(*) FROM товары WHERE цена > 100;
```

Такой запрос вернёт **одну строку с одним числом** — не список
товаров, а именно счёт. Это ровно наш счётчик из цикла,
только считать будет база.

### SQL в Access

Access понимает SQL, но свой диалект — отличий немного, и все они
в мелочах:

| Везде | В Access |
|---|---|
| `LIMIT 3` в конце запроса | `TOP 3` сразу после `SELECT` |
| `LIKE 'Мо%'` — любое число символов | `LIKE 'Мо*'` |
| `LIKE 'М_ко'` — ровно один символ | `LIKE 'М?ко'` |
| текст в `'одинарных'` кавычках | можно и в `"двойных"` |

Запрос в Access можно не писать вовсе, а собрать мышкой
в **конструкторе** — Access сам превратит его в SQL. Этим
и займёмся: собирать — в конструкторе, проверять — в режиме SQL.

## Практическая работа. Запросы к магазину

[Скачайте базу «Магазин»](https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/files/access/g09-23-magazin.accdb) и откройте
её в **Access**. Если сверху жёлтая полоса — **Включить
содержимое**. В базе одна таблица «Товары» на 20 строк
(в списке слева есть и готовые запросы — ими сверяйтесь,
но сначала соберите свои).

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/access/g09-23-tovary.png" width="900" alt="Таблица «Товары»">

*Таблица «Товары»*

Поле «Скидка» заполнено не у всех: пустая ячейка — это `NULL`,
«значения нет».

### Шаг 1. Конструктор запросов

**Создание → Конструктор запросов**. В окне **Добавление таблицы**
выберите «Товары» → **Добавить** → **Закрыть**.

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/access/g09-23-dobavlenie.png" width="900" alt="Окно «Добавление таблицы»">

*Окно «Добавление таблицы»*

Внизу — **бланк запроса**. Каждый столбец бланка — одно поле.
Строки бланка — это части SQL:

| Строка бланка | Часть SQL |
|---|---|
| Поле | что показать — `SELECT` |
| Имя таблицы | откуда — `FROM` |
| Сортировка | `ORDER BY` |
| Вывод на экран | показывать ли поле (условие можно ставить и по скрытому) |
| Условие отбора | `WHERE` |
| или | ещё одно условие через `OR` |

### Шаг 2. Условие и сортировка

Вопрос: **какие товары дороже 100 ₽**, от дорогих к дешёвым?

Дважды щёлкните по полям «Название», «Категория», «Цена» — они
встанут в бланк. В столбце «Цена»: **Условие отбора** — `>100`,
**Сортировка** — *по убыванию*.

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/access/g09-23-uslovie.png" width="900" alt="Условие и сортировка">

*Условие и сортировка*

**Выполнить** (красный «!»):

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/access/g09-23-uslovie-rezultat.png" width="900" alt="Товары дороже 100">

*Товары дороже 100*

Внизу окна — «Запись: 1 из …»: это сколько строк вернул запрос.
Сохраните (`Ctrl` + `S`) как `Дороже 100`. Вернуться в конструктор —
**Главная → Режим → Конструктор**.

### Шаг 3. И — в одной строке

Вопрос: **молочное дешевле 100 ₽**. Два условия, оба обязательны.

Новый запрос с теми же полями. В «Категория» — `молочное`,
в «Цена» — `<100`, **в одной строке** «Условие отбора». Условия
в одной строке соединяются через **И**.

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/access/g09-23-i.png" width="900" alt="Два условия через И">

*Два условия через И*

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/access/g09-23-i-rezultat.png" width="900" alt="Молочное дешевле 100">

*Молочное дешевле 100*

### Шаг 4. ИЛИ — в разных строках

Вопрос: **все напитки и вся выпечка**. Здесь нужно «или»: товар
не может быть сразу и тем, и другим.

В столбце «Категория»: в строке **Условие отбора** — `напитки`,
в строке **или** ниже — `выпечка`. Access перенесёт это в одну
ячейку как `"напитки" Or "выпечка"` — это то же самое.

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/access/g09-23-ili.png" width="900" alt="Условие через ИЛИ">

*Условие через ИЛИ*

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/access/g09-23-ili-rezultat.png" width="900" alt="Напитки или выпечка">

*Напитки или выпечка*

> Частая ошибка: написать `напитки` и `выпечка` **в одну строку**
> разных столбцов или через And. Тогда Access ищет товар, который
> одновременно и напиток, и выпечка, — и находит ноль строк.

### Шаг 5. Шаблон и пустые значения

**Название начинается на «К».** В «Название» — условие `Like "К*"`.
Звёздочка — любые символы после «К». Если написать просто `К*`,
Access сам допишет `Like`.

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/access/g09-23-like.png" width="900" alt="Условие Like">

*Условие Like*

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/access/g09-23-like-rezultat.png" width="900" alt="Товары на «К»">

*Товары на «К»*

**Товары без скидки.** В «Скидка» — `Is Null`. Не `=0` и не пусто:
пустое условие означает «условия нет», а ноль — это число.

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/access/g09-23-null.png" width="900" alt="Условие Is Null">

*Условие Is Null*

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/access/g09-23-null-rezultat.png" width="900" alt="Товары без скидки">

*Товары без скидки*

Ещё два полезных условия: `Between 50 And 100` — от 50 до 100
включительно; `In ("чай";"кофе")` — одно из списка.

### Шаг 6. Подсчёты: итоги

Вопрос: **сколько товаров в каждой категории и какая у них
средняя цена**?

Новый запрос: поля «Категория», «Название», «Цена». Нажмите
на ленте **Итоги** (Σ). В бланке появится строка **Групповая
операция**:

* «Категория» — *Группировка*;
* «Название» — *Count* (сколько);
* «Цена» — *Avg* (среднее).

Чтобы столбцы в ответе назывались по-человечески, пишите имя
перед полем через двоеточие: `Товаров: Название`, `Средняя цена: Цена`.

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/access/g09-23-itogi.png" width="900" alt="Итоги">

*Итоги*

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/access/g09-23-itogi-rezultat.png" width="900" alt="По категориям">

*По категориям*

Это `COUNT` и `AVG` с `GROUP BY` — подробно группировку разбирают
в 11 классе.

### Шаг 7. Вычисляемое поле

Вопрос: **на сколько рублей каждого товара лежит на складе**?
Такого поля в таблице нет — посчитаем. В пустом столбце бланка
в строке «Поле» напишите:

```
Стоимость: [Цена]*[Количество]
```

Имена полей — в квадратных скобках. Сортировка — по убыванию.

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/access/g09-23-vychislenie.png" width="900" alt="Вычисляемое поле">

*Вычисляемое поле*

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/access/g09-23-vychislenie-rezultat.png" width="900" alt="Стоимость по товарам">

*Стоимость по товарам*

### Шаг 8. Смотрим SQL

Откройте готовый запрос «Три самых дорогих» и переключитесь
в **Режим SQL** (**Главная → Режим**):

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/access/g09-23-sql-top.png" width="900" alt="TOP 3 в режиме SQL">

*TOP 3 в режиме SQL*

`TOP 3` — это `LIMIT 3` по-аксессовски. В режиме SQL запрос можно
и писать с нуля: наберите текст и нажмите **Выполнить**.
Посмотрите в режиме SQL каждый свой запрос — узнаете в нём
`WHERE`, `AND`, `OR`, `ORDER BY` из теории.

## Проверяем себя

Все ответы — из запросов к базе «Магазин».

Дальше вы пишете запросы сами. В каждой задаче нужно дописать
функцию: она получает `курсор`, выполняет запрос и возвращает
`fetchall()` — список кортежей.

База та же, что в примерах: таблица `товары` с полями
`id, название, категория, цена, количество`.

### Задача 1. Сколько дороже 100

Сколько строк вернул запрос «Дороже 100»?

In [ ]:
#@title 🧩 Задача 1. Цена больше 100 { display-mode: "form" }
#@markdown Впиши число
дороже = 0 #@param {type:"integer"}

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("1", дороже, "4a44dc15364204a8",
         hint="Условие >100 в столбце «Цена».")

### Задача 2. Самый дорогой

Какой товар первым стоит в запросе «Дороже 100» — то есть самый
дорогой?

In [ ]:
#@title 🧩 Задача 2. Первый после сортировки { display-mode: "form" }
#@markdown Впиши название
дорогой = "" #@param {type:"string"}

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("2", дорогой, "8e3d590fb7b6665b",
         hint="Сортировка по убыванию цены.")

### Задача 3. И

Сколько молочных товаров дешевле 100 ₽?

In [ ]:
#@title 🧩 Задача 3. Два условия { display-mode: "form" }
#@markdown Впиши число
молочных = 0 #@param {type:"integer"}

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("3", молочных, "4b227777d4dd1fc6",
         hint="Оба условия — в одной строке бланка.")

### Задача 4. Шаблон

Сколько товаров начинается на букву «К»?

In [ ]:
#@title 🧩 Задача 4. Like "К*" { display-mode: "form" }
#@markdown Впиши число
на_к = 0 #@param {type:"integer"}

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("4", на_к, "7902699be42c8a8e",
         hint="Звёздочка после буквы.")

### Задача 5. Итоги

Какая средняя цена у сладостей? Через запятую, как в ответе
запроса, без «₽».

In [ ]:
#@title 🧩 Задача 5. Avg { display-mode: "form" }
#@markdown Впиши число
средняя = "" #@param {type:"string"}

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("5", средняя, "874274cc7ddb2481",
         hint="Итоги: группировка по категории, Avg по цене.")

### Задача 6. Слово для условия

Какое слово начинает часть запроса, где задаётся условие отбора
строк? Запишите заглавными буквами.

In [ ]:
#@title 🧩 Задача 6. Слово запроса { display-mode: "form" }
#@markdown Впиши слово
слово = "" #@param {type:"string"}

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("6", слово, "43f94244d00ce481",
         hint="По-английски «где».")

## Домашнее задание

### Домашнее задание 1. Восемь запросов

В базе «Магазин» соберите и сохраните восемь запросов. Имя
запроса — номер и суть: `1 Напитки`, `2 Дешевле 100` и так далее.

1. все товары категории «напитки», по алфавиту;
2. товары дешевле 100 ₽ — по убыванию цены;
3. три самых дорогих товара — в режиме SQL, через `TOP 3`;
4. товары от 100 до 200 ₽ включительно — через `Between`;
5. товары со скидкой (не пустой): условие `Is Not Null`;
6. сколько товаров в каждой категории — итоги, Count;
7. товары, которых на складе меньше 10 штук, — с полями
   «Название» и «Количество», по возрастанию количества;
8. **общая** стоимость склада — вычисляемое поле
   `[Цена]*[Количество]` и итоги с операцией **Sum**, без других
   полей. Ответ — одно число.

Для каждого запроса откройте режим SQL и перепишите текст
в тетрадь. Сдайте учителю файл базы с восемью запросами.

### Домашнее задание 2. Самый дешёвый

Какой товар в магазине самый дешёвый? Соберите запрос
с сортировкой по возрастанию цены.

In [ ]:
#@title 🏠 Домашнее 2. Минимальная цена { display-mode: "form" }
#@markdown Впиши название
дешёвый = "" #@param {type:"string"}

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("дз2", дешёвый, "daa090af00387ad4",
         hint="Первая строка после сортировки по возрастанию.")

### Домашнее задание 2а. Стоимость склада

Запрос 8 из первого задания даёт одно число — сколько стоит весь
склад. Впишите его без «₽» и пробелов.

In [ ]:
#@title 🏠 Домашнее 2а. Весь склад { display-mode: "form" }
#@markdown Впиши число
склад = 0 #@param {type:"integer"}

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("дз2а", склад, "22452caad50ec2e5",
         hint="Итоги: Sum по полю Стоимость: [Цена]*[Количество].")

### Домашнее задание 3. Порядок по умолчанию

Если написать `ORDER BY цена` без дополнительных слов,
в каком порядке отсортируется результат?

In [ ]:
#@title 🏠 Домашнее 3. Порядок сортировки { display-mode: "form" }
#@markdown Выбери ответ
порядок = "выбери ответ" #@param ["выбери ответ", "по возрастанию", "по убыванию", "случайный"]

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("дз3", порядок, "715c407e83e0e848",
         hint="Для убывания пришлось бы дописать DESC.")

---

### Любопытно

SQL придумали в IBM в начале 1970-х и назвали SEQUEL — «структурный
английский язык запросов». Название пришлось менять: оно уже было
зарегистрировано британской авиастроительной фирмой. Осталось SQL,
но произносят его до сих пор двояко — «эс-кью-эль» и «сиквел».

Языку полвека, и за это время сменились поколения языков
программирования, а SQL остался почти тем же. Запрос, написанный
в 1975 году, сегодня выполнится без единой правки — похвастаться
таким может мало кто.

---

## Отчёт по уроку

Запусти ячейку ниже, когда решишь задачи. Результат уйдёт учителю автоматически.

In [ ]:
si.report()

---

[← Урок 22](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-22.ipynb) · [⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/index.ipynb) · [🏠 Ко всем классам](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/index.ipynb) · [Урок 24 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-24.ipynb)